# TES 多模态回归数据 -> SFT DataLoader 文本验证
这个 notebook 只验证：
1. `tes_multimodal_sft_dataloader.py` 可以正确读取 `npz + json`
2. 输出样本是标准 SFT `messages` 格式
3. `user/assistant` 文本内容可读且结构正确（不做导出）


In [1]:
from pathlib import Path
import json
import sys
import os
sys.path.append('/home/yichenglu/zhongkong/code/luyicheng/Inferential_Data_Generation/inferential_data_generation')

from multimodal_regression_data_construction.tes_multimodal_sft_dataloader import (
    TESMultimodalRegressionSFTDataset,
    build_sft_dataloader,
    export_sft_jsonl,
)

# 你可以改成 test 文件
NPZ_PATH = Path("/home/yichenglu/zhongkong/code/luyicheng/Inferential_Data_Generation/data/regress_data/tes_regression_seq96_pred1_train.npz")
JSON_PATH = Path("/home/yichenglu/zhongkong/code/luyicheng/Inferential_Data_Generation/data/regress_data/tes_regression_seq96_pred1_train.json")
OUT_PATH = Path("/home/yichenglu/zhongkong/code/luyicheng/Inferential_Data_Generation/data/regress_data/tes_regression_seq96_pred1_train_sft_demo.jsonl")

assert NPZ_PATH.exists(), f"文件不存在: {NPZ_PATH}"
assert JSON_PATH.exists(), f"文件不存在: {JSON_PATH}"
print("输入文件检查通过")


输入文件检查通过


In [2]:
# 1) 直接构建 Dataset

dataset = TESMultimodalRegressionSFTDataset(
    npz_path=NPZ_PATH,
    json_path=JSON_PATH,
    system_prompt="You are a regression assistant. Return numeric predictions only.",
    series_mode="raw",  # 可选: "stats" 或 "raw"
)

print("dataset size:", len(dataset))
sample0 = dataset[0]
print("sample keys:", sample0.keys())
print("message turns:", len(sample0["messages"]))
print(json.dumps(sample0, ensure_ascii=False)[:1000])


dataset size: 40500
sample keys: dict_keys(['sample_id', 'messages'])
message turns: 3
{"sample_id": 0, "messages": [{"role": "system", "content": "You are a regression assistant. Return numeric predictions only."}, {"role": "user", "content": "Task: Predict regression targets for `xmeas_40` and `xmeas_41` from numerical time-series inputs and text covariates. Given a single sample, predict the next-step product quality using two inputs: a time-series window (L=96) of key variables, and one-line-per-variable text summaries for the remaining variables (covering trend, fluctuation, and change rate). The output must be JSON only, containing no explanations, units, or extra fields, following the fixed schema: `{\"xmeas_40\": <float>, \"xmeas_41\": <float>}`, where both values are rounded to two decimal places.\nScene: This task is based on process data from the Tennessee Eastman Process (TEP). This process comprises various units—including feed systems, reactors, separators, strippers, and

In [3]:
# 2) 严格校验 SFT 格式

def validate_sft_item(item: dict):
    assert isinstance(item, dict)
    assert "messages" in item
    msgs = item["messages"]
    assert isinstance(msgs, list)
    assert len(msgs) == 3

    expected_roles = ["system", "user", "assistant"]
    for i, role in enumerate(expected_roles):
        assert msgs[i]["role"] == role, f"第{i}轮 role 错误: {msgs[i]['role']}"
        assert isinstance(msgs[i]["content"], str)
        assert len(msgs[i]["content"].strip()) > 0

validate_sft_item(sample0)
print("单样本 SFT 格式校验通过")


单样本 SFT 格式校验通过


In [4]:
# 3) 构建 DataLoader 并检查 batch

dataset2, dataloader = build_sft_dataloader(
    npz_path=NPZ_PATH,
    json_path=JSON_PATH,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    series_mode="raw",
)

batch = next(iter(dataloader))
print("batch keys:", batch.keys())
print("batch sample_ids:", batch["sample_ids"])
print("batch size:", len(batch["messages"]))

for i, messages in enumerate(batch["messages"]):
    item = {"messages": messages}
    validate_sft_item(item)
print("batch SFT 格式校验通过")


batch keys: dict_keys(['sample_ids', 'messages'])
batch sample_ids: [0, 1, 2, 3]
batch size: 4
batch SFT 格式校验通过


In [5]:
# 4) 直接输出完整 messages

import json

# 保留最小必要校验
validate_sft_item(sample0)

print(json.dumps(sample0["messages"], ensure_ascii=False, indent=2))


[
  {
    "role": "system",
    "content": "You are a regression assistant. Return numeric predictions only."
  },
  {
    "role": "user",
    "content": "Task: Predict regression targets for `xmeas_40` and `xmeas_41` from numerical time-series inputs and text covariates. Given a single sample, predict the next-step product quality using two inputs: a time-series window (L=96) of key variables, and one-line-per-variable text summaries for the remaining variables (covering trend, fluctuation, and change rate). The output must be JSON only, containing no explanations, units, or extra fields, following the fixed schema: `{\"xmeas_40\": <float>, \"xmeas_41\": <float>}`, where both values are rounded to two decimal places.\nScene: This task is based on process data from the Tennessee Eastman Process (TEP). This process comprises various units—including feed systems, reactors, separators, strippers, and recycle/cooling loops—and is characterized by significant coupling, time delays, and nonl